# 03 · CTransPath spatial and RGB features

Read native H&E images, extract frozen pretrained CTransPath features, and save spatial tokens and regional RGB values.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import h5py
import numpy as np
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/02_he_prediction").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "he_reference_v2")
OUT = ROOT / "outputs/he_prediction" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
FEATURE_DIR = OUT / "features"
SLIDE_ROWS = [
    ("proseg1", 984, 1784, 27761, 50000, "proseg1.tif", "proseg1.h5"),
    ("proseg2", 824, 1784, 23140, 50000, "proseg2.tif", "proseg2.h5"),
    ("proseg3", 856, 1784, 23981, 50000, "proseg3.tif", "proseg3.h5"),
    ("proseg4", 912, 1784, 25667, 50000, "proseg4.tif", "proseg4.h5"),
    ("proseg5", 896, 1784, 25108, 50000, "proseg5.tif", "proseg5.h5"),
    ("proseg6", 912, 1784, 25706, 50000, "proseg6.tif", "proseg6.h5"),
    ("proseg7", 1624, 3472, 45603, 97266, "proseg7.tif", "proseg7.h5"),
    ("proseg8", 1480, 3472, 41510, 97267, "proseg8.tif", "proseg8.h5"),
    ("proseg9", 1624, 3472, 45603, 97262, "proseg_9.tif", "proseg_9.h5"),
    ("proseg10", 1624, 3472, 45605, 97259, "proseg_10.tif", "proseg_10.h5"),
]
SLIDES = []
for (
    name,
    height,
    width,
    native_height,
    native_width,
    image_name,
    feature_name,
) in SLIDE_ROWS:
    index = name.removeprefix("proseg")
    SLIDES.append(
        {
            "name": name,
            "h5ad_path": str(
                ROOT / "data/he_aligned_h5ad" / f"proseg_{index}_aligned.h5ad"
            ),
            "h5_path": str(FEATURE_DIR / feature_name),
            "he_tif_path": str(ROOT / "data/he_slides" / image_name),
            "niche_label_path": str(
                ROOT
                / "data/he_export_2026_10_09/labels"
                / f"Proseg{index}"
                / "label_niche_uint16.dat"
            ),
            "domain_label_path": str(
                ROOT
                / "data/he_export_2026_10_09/labels"
                / f"Proseg{index}"
                / "label_domain_uint16.dat"
            ),
            "label_shape": [height, width],
            "H": height,
            "W": width,
            "level0_shape": [native_height, native_width],
            "he_shape": [height * 28, width * 28],
            "region_size": 28,
        }
    )
TRAIN_SLIDES = SLIDES[:8]
VALIDATION_SLIDES = SLIDES[8:]
TOKEN_NUM = 49
FEAT_DIM = 768
CHECKPOINT = ROOT / "models/he_prediction/export_2026_10_09/ctranspath.pth"

In [ ]:
from spagc._io import require_files
from spagc.histology.features import (
    extract_ctranspath_from_svs,
    h5_feature_file_is_complete,
    load_ctranspath,
)
from spagc.histology.metadata import prepare_slides

The slide table lists native images and output feature names. Its columns are name, label height, label width, native height, native width, image filename and feature filename.

In [ ]:
IMAGE_DIR = ROOT / "data/he_slides"
STRIDE, BATCH_SIZE = (112, 256)
require_files(CHECKPOINT)
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(IMAGE_DIR)
cohort = prepare_slides(TRAIN_SLIDES + VALIDATION_SLIDES, root=ROOT)
for item in cohort:
    require_files(item["he_tif_path"])

## 2. Restore the frozen CTransPath encoder

Strictly restore the supplied pretrained checkpoint. The encoder supplies spatial tokens and is not trained in this step.

Use the original convolutional stem in the Swin-based CTransPath architecture.

Build CTransPath and strictly restore the supplied pretrained weights; keep the encoder frozen.

In [ ]:
backbone = load_ctranspath(str(CHECKPOINT), DEVICE)

## 3. Extract and store slide features

Extract overlapping native-resolution tiles. Write token features, RGB region features and tile positions directly to bounded-memory HDF5 files; reuse only files that pass completion and geometry checks.

In [ ]:
extracted = []
try:
    for item in cohort:
        output_h5 = Path(item["h5_path"])
        if h5_feature_file_is_complete(output_h5, item["he_tif_path"], stride=STRIDE):
            print("Reusing complete feature file:", output_h5)
            continue
        extracted.append(
            extract_ctranspath_from_svs(
                item["he_tif_path"],
                str(output_h5),
                model=backbone,
                device=DEVICE,
                batch_size=BATCH_SIZE,
                stride=STRIDE,
            )
        )
finally:
    del backbone
print(extracted)

## 4. Check the stored feature arrays

Verify 49 tokens with 768 channels, RGB regions with shape 3 by 8 by 8, and matching tile coordinates.

In [ ]:
paths = sorted(FEATURE_DIR.glob("*.h5"))
assert paths, "No feature files were produced."
for path in paths:
    with h5py.File(path) as handle:
        assert handle["patch_features"].shape[1:] == (49, 768)
        assert handle["rgb_regions"].shape[1:] == (3, 8, 8)
        assert handle.attrs["extraction_complete"]
        assert np.isfinite(handle["patch_features"][:1]).all()
        assert len(handle["coords"]) == len(handle["patch_features"])
        print(path.name, dict(handle.attrs), handle["patch_features"].shape)